In [ ]:
import os
from flask import Flask, redirect, render_template, request
from PIL import Image
import numpy as np
import torch
import pandas as pd

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pyngrok

In [ ]:
from flask import Flask
from pyngrok import ngrok

In [ ]:
pip install keras --upgrade

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

In [ ]:
import cv2

In [2]:
import chardet

with open('/content/drive/MyDrive/Amedi/plantinfo.csv', 'rb') as f:
    result = chardet.detect(f.read())

print(result['encoding'])


Windows-1252


In [ ]:
plant_df = pd.read_csv('/content/drive/MyDrive/Amedi/plantinfo.csv', encoding='Windows-1252')

In [ ]:
model_path = '/content/drive/MyDrive/Amedi/mediplant.h5'  # Update with the correct path
cnn = tf.keras.models.load_model(model_path)

In [ ]:
import pandas as pd
from io import BytesIO

In [ ]:
def prediction(image_stream):
    # Assuming you process the image stream as before to get the predicted index
    image = Image.open(image_stream)
    image = image.convert("RGB")
    image = image.resize((64, 64))
    image = tf.keras.preprocessing.image.load_img(image_stream,target_size=(64,64))
    input_arr = tf.keras.preprocessing.image.img_to_array(image)
    input_arr = np.array([input_arr])  # Convert single image to a batch.
    predictions = cnn.predict(input_arr)
    index = np.argmax(predictions)  # Get the predicted class index
    return index

In [ ]:
ngrok.set_auth_token("2Yerk4RNRoBEYzBuHbOtRiAVLbo_288gvqWMZp8YkmmbqmeEg")

In [ ]:
from flask import Flask
from pyngrok import ngrok

app = Flask(__name__, template_folder='/content/drive/MyDrive/Amedi/templates')

@app.route('/')
def login():
    return render_template('login.html')

@app.route('/home', methods=['GET', 'POST'])
def home():
    if request.method == 'POST':
        # Handle login logic here
        return render_template('home.html')

@app.route('/Contact-us')
def contact():
    return render_template('contact-us.html')

@app.route('/index')
def index():
    return render_template('index.html')

@app.route('/submit', methods=['GET', 'POST'])
def submit():
    try:
        if request.method == 'POST':
            image = request.files['image']

            # Ensure an image is uploaded
            if not image:
                return "No image uploaded", 400

            # Read the uploaded image as a BytesIO stream
            img_stream = BytesIO(image.read())

            # Pass the image stream to the prediction function
            pred = prediction(img_stream)

            # Fetch the plant information based on the predicted index from the CSV DataFrame
            plant_info = plant_df.iloc[pred]  # Get the row corresponding to the predicted index
            title = plant_info['Plant_name']
            Sci_name = plant_info['scientific_name']
            clas = plant_info['class ']
            Order = plant_info['order']
            description = plant_info['description']
            advant = plant_info['advantages']
            disadvant = plant_info['disadvantages']
            url = plant_info['Url']

            # Render the template with the plant info
            return render_template(
                'submit.html',
                title=title,
                image=url,
                scientific=Sci_name,
                Class=clas,
                order=Order,
                desc=description,
                advantage=advant,
                disadvantage=disadvant,
            )
    except Exception as e:
        # Log and return the error for debugging
        print(f"Error: {e}")
        return f"An error occurred: {e}", 500



if __name__ == "__main__":
    # Explicitly define port and protocol
    public_url = ngrok.connect(5000, "http")
    print(f"Public URL: {public_url}")

    # Run the Flask app
    app.run(host="0.0.0.0", port=5000)


Public URL: NgrokTunnel: "https://b3d4-34-23-118-147.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:46:37] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:46:38] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:46:45] "POST /home HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:46:52] "GET /index HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step


INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:47:11] "POST /submit HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:49:20] "GET /index HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


INFO:werkzeug:127.0.0.1 - - [07/Feb/2025 09:49:31] "POST /submit HTTP/1.1" 200 -
